# SAGE — end-to-end training on Colab

Clones the sage repo, installs dependencies, aggregates the seven training datasets, trajectorizes, trains, and exports to ONNX.

**Before running:**
1. **Runtime → Change runtime type → GPU** (L4 or A100 if you have Pro/Pro+)
2. Get a HuggingFace token at <https://huggingface.co/settings/tokens>
3. In Colab, open the key icon in the left sidebar and add a secret named `HF_TOKEN` with your token as the value. Toggle "Notebook access".
4. Optional: mount Google Drive for persistent checkpoints (first cell).

## 1. Mount Drive (recommended — checkpoints survive disconnects)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Checkpoints + exported artifacts will live here.
WORK_DIR = '/content/drive/MyDrive/sage'

import os
os.makedirs(WORK_DIR, exist_ok=True)
print(f'working in: {WORK_DIR}')

## 2. Clone the repo

In [ ]:
%cd /content
!rm -rf sage
!git clone https://github.com/MegalithOfficial/sage.git
%cd sage

## 3. Install dependencies

Colab ships with PyTorch already. We only need the training extras.

In [ ]:
!pip install -q -e '.[train,export]'

## 4. HuggingFace auth

In [ ]:
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

## 5. Quick GPU sanity check

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 6. Aggregate datasets (~15–30 min first run)

Pulls all 7 sources at 50k examples each. Writes splits to `data/processed/`. Inspect `stats.json` afterwards.

In [ ]:
!python -m training.data.aggregate \
    --out data/processed \
    --sources all \
    --limit-per-source 50000

!cat data/processed/stats.json | python -c "import json,sys; s=json.load(sys.stdin); print(json.dumps(s['splits']['train']['per_category_positive'], indent=2))"

## 7. Trajectorize

In [ ]:
for split in ['train', 'val', 'test']:
    !python -m training.data.trajectorize_cli \
        --in  data/processed/{split}.jsonl \
        --out data/processed/{split}.traj.jsonl

## 8. Train

Hyperparameters auto-adjust to the available VRAM.
Checkpoints stream to Drive so disconnects don't lose progress.

Expected time:
- **T4 16 GB** (Colab free / Pro fallback): ~4–6 hrs
- **L4 24 GB** (Pro default): ~2–3 hrs
- **A100 40 GB** (Pro+ / sometimes Pro): ~45 min – 1.5 hrs

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 35:
    batch_size, grad_accum, max_length = 32, 1, 1024
elif vram_gb >= 22:
    batch_size, grad_accum, max_length = 16, 2, 1024
elif vram_gb >= 14:
    batch_size, grad_accum, max_length = 8, 4, 512
else:
    batch_size, grad_accum, max_length = 4, 8, 256

print(f'VRAM {vram_gb:.0f} GB → batch={batch_size} grad_accum={grad_accum} max_length={max_length}')
print(f'Effective batch size: {batch_size * grad_accum}')

CKPT_DIR = f'{WORK_DIR}/checkpoints/sage-v1'
!mkdir -p {CKPT_DIR}

!python -m training.train \
    --train data/processed/train.traj.jsonl \
    --val   data/processed/val.traj.jsonl \
    --out   {CKPT_DIR} \
    --epochs 3 \
    --batch-size {batch_size} \
    --grad-accum {grad_accum} \
    --max-length {max_length} \
    --num-workers 4 \
    --log-every 100

## 9. Export to ONNX + INT8

In [ ]:
ART_DIR = f'{WORK_DIR}/artifacts'
!mkdir -p {ART_DIR}

!python -m training.export_onnx \
    --checkpoint {CKPT_DIR}/best.pt \
    --max-length {max_length} \
    --out-fp32 {ART_DIR}/sage-v1-fp32.onnx \
    --out-int8 {ART_DIR}/sage-v1-int8.onnx

!ls -lh {ART_DIR}

## 10. Quick smoke test on the exported model

In [ ]:
from sage.inference import Sage
model = Sage.from_onnx(f'{ART_DIR}/sage-v1-int8.onnx')

tests = [
    'how do I bake bread?',
    'I hate all [slur], they ruin everything',
    'what are good study techniques',
    'I want to kill myself',
]
for t in tests:
    r = model.check(t)
    flagged = [c for c, v in r.categories.items() if v.flagged]
    print(f'flagged={flagged} \t text={t!r}')